# Profiling computations

The below code loads and encodes the data. Then the profiling module cProfile together with a
graphical viewer [SnakeViz](https://jiffyclub.github.io/snakeviz/) is used to identify the
functions/methods executed most frequently and where the execution time is used.

It uses cell magic (%%snakeviz) and the visualization should work inline when executed as juputer
notebook.


In [ ]:
# Install the required packages below

# !pip install more-itertools==8.14.0
# !pip install numpy==1.25.2
# !pip install pandas==2.1.0
# !pip install scikit-learn==1.3.1
# !pip install scikit-rough==0.2.0
# !pip install snakeviz==2.2.0
# !pip install xgboost==2.0.0

In [1]:
import pathlib
import cProfile
import pstats
from functools import partial

import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from skrough.algorithms.reducts import get_approx_reduct_greedy_heuristic
from skrough.disorder_measures import gini_impurity

In [2]:
DATA_DIR = pathlib.Path("../../data")
RESULTS_DIR = pathlib.Path("results")

data = pd.read_csv(DATA_DIR / "seismic_training_data_features_discretized.csv")
target = pd.read_csv(DATA_DIR / "seismic_training_labels.csv", header=None)[0]

# drop main_working_id column
data.drop("main_working_id", axis=1, inplace=True)

# encode data and target into the form of integers
data = OrdinalEncoder().fit_transform(data)
target = LabelEncoder().fit_transform(target)

In [3]:
from sklearn.model_selection import train_test_split

data_small, _, target_small, _ = train_test_split(
    data, target, train_size=0.2, random_state=42
)
print(f"Data shape: {data_small.shape}")

Data shape: (26630, 738)


# Profiling the greedy reducts computation

We use ``n_jobs=1`` to ease profiling info ingestion - it may take some time to finish...

In [4]:
def profile_reducts(reduct_func, output_file):
    profiler = cProfile.Profile()
    profiler.enable()
    reduct_func()
    profiler.disable()
    stats = pstats.Stats(profiler)
    stats.dump_stats(output_file)

In [ ]:
greedy_heuristic_lazy = partial(
    get_approx_reduct_greedy_heuristic,
    x=data_small,
    y=target_small,
    disorder_fun=gini_impurity,
    epsilon=0.01,
    candidates_count=100,
    n_reducts=5,
    n_jobs=1,
    group_index_class="lazy",
)

profile_reducts(greedy_heuristic_lazy, RESULTS_DIR / "greedy_heuristic_lazy.prof")

In [5]:
greedy_heuristic_pure = partial(
    get_approx_reduct_greedy_heuristic,
    x=data,
    y=target,
    disorder_fun=gini_impurity,
    epsilon=0.01,
    candidates_count=100,
    n_reducts=20,
    n_jobs=1,
    group_index_class="pure",
)

profile_reducts(greedy_heuristic_pure, RESULTS_DIR / "greedy_heuristic_pure.prof")

In [ ]:
greedy_heuristic_numba = partial(
    get_approx_reduct_greedy_heuristic,
    x=data,
    y=target,
    disorder_fun=gini_impurity,
    epsilon=0.01,
    candidates_count=100,
    n_reducts=20,
    n_jobs=1,
    group_index_class="numba",
)

profile_reducts(greedy_heuristic_numba, RESULTS_DIR / "greedy_heuristic_numba.prof")